In [335]:
print(unknowns[['First arrest rhythm', 'Cardiac rhythm on arrival at ED']])

     First arrest rhythm  Cardiac rhythm on arrival at ED
1307                 NaN                              PEA
1390                 NaN                         ASYSTOLE
1564             UNKNOWN                         ASYSTOLE
1635                 NaN                              PEA
1639                 NaN                         ASYSTOLE
1786             UNKNOWN                         ASYSTOLE
2011             UNKNOWN  SINUS OR OTHER PERFUSING RHYTHM
2169                 NaN                         ASYSTOLE


# Install libraries

In [336]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import statsmodels.api as sm
import graphviz
from IPython.display import display
from scipy.optimize import curve_fit
from dotenv import load_dotenv

from pathlib import Path

# Setting up file paths

In [337]:
CURRENT_DIRECTORY = Path(os.getcwd()).resolve()

# Find project root by walking up until the "datasets" folder is found
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

# Set the base dataset path
BASE_DATASET_PATH = PROJECT_ROOT / "datasets"

# Define file paths directly inside the datasets folder
DECRYPTED_FILE_PATH = BASE_DATASET_PATH / "PAROS_Dataset.csv"
CLEANED_DATASET_PATH = BASE_DATASET_PATH / "PAROS_Dataset_Cleaned_2010_2017.csv"

print("CURRENT_DIRECTORY:", CURRENT_DIRECTORY)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DECRYPTED_FILE_PATH exists:", DECRYPTED_FILE_PATH.exists())
display(DECRYPTED_FILE_PATH)

CURRENT_DIRECTORY: /Users/axlee/Desktop/Singhealth/OHCA/src/Cleaning PAROS dataset
PROJECT_ROOT: /Users/axlee/Desktop/Singhealth/OHCA
DECRYPTED_FILE_PATH exists: True


PosixPath('/Users/axlee/Desktop/Singhealth/OHCA/datasets/PAROS_Dataset.csv')

# Decrypting the file

In [338]:
print(f"Attempting to open file: {DECRYPTED_FILE_PATH.name}...")

try:

    df = pd.read_csv(DECRYPTED_FILE_PATH)
    print("✅ PAROS dataset successfully decrypted and loaded!")
    
    # Show the first 3 rows to confirm
    display(df.head())
    print(len(df))

except FileNotFoundError:
    print(f"❌ Error: Could not find the file at {DECRYPTED_FILE_PATH}. Please check the path and filename.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

Attempting to open file: PAROS_Dataset.csv...
✅ PAROS dataset successfully decrypted and loaded!


/var/folders/y3/l8mcqj111md_2kpwgj8zhk0w0000gn/T/ipykernel_81040/1203111889.py:5: DtypeWarning: Columns (9,62,63,64,65,67,71,72,78,86,91,92,94,107,111,112,114,121,123,124) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DECRYPTED_FILE_PATH)


,Case #,Country,City,Site #,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,...,EQ-5D Mobility,EQ-5D Self-care,EQ-5D Usual activities,EQ-5D Pain/Discomfort,EQ-5D Anxiety/Depression,EQ5D index,EQ-5D VAS,General Comments,Date Created,Date Last Saved
0,SGSIN0213,SG,SIN,2.0,EMS,4/1/2010,470146.0,NaN,Home Residence,HDB Level 7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,2/22/2011
1,SGSIN0218,SG,SIN,2.0,EMS,4/1/2010,520926.0,NaN,Home Residence,HDB Level 2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,2/22/2011
2,SGSIN6480,SG,SIN,6.0,EMS,4/1/2010,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,4/18/2012
3,SGSIN5332,SG,SIN,5.0,EMS,4/2/2010,680626.0,NaN,Home Residence,HDB Level 7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,4/13/2012
4,SGSIN0214,SG,SIN,2.0,EMS,4/3/2010,468963.0,NaN,Place of Recreation,East Coast Park NSC Carpark,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,2/22/2011


28666


# Fill missing values with NaN

In [339]:
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))

missing_tokens = ["", " ", "  ", "N/A", "n/a", "NA", "na", "NULL", "null", "None", "none", "-"]
df = df.replace(missing_tokens, np.nan)

display(df.head(3))
print(len(df))

,Case #,Country,City,Site #,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,...,EQ-5D Mobility,EQ-5D Self-care,EQ-5D Usual activities,EQ-5D Pain/Discomfort,EQ-5D Anxiety/Depression,EQ5D index,EQ-5D VAS,General Comments,Date Created,Date Last Saved
0,SGSIN0213,SG,SIN,2.0,EMS,4/1/2010,470146.0,NaN,Home Residence,HDB Level 7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011
1,SGSIN0218,SG,SIN,2.0,EMS,4/1/2010,520926.0,NaN,Home Residence,HDB Level 2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011
2,SGSIN6480,SG,SIN,6.0,EMS,4/1/2010,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4/18/2012


28666


In [340]:
df['Year'] = pd.to_datetime(df['Date of Incident'], errors='coerce').dt.year
print(df['Year'].value_counts(dropna=False).sort_index())

Year
2010.0    1081
2011.0    1377
2012.0    1440
2013.0    1736
2014.0    2038
2015.0    2372
2016.0    2505
2017.0    2841
2018.0    2975
2019.0    3233
2020.0    3431
2021.0    1513
NaN       2124
Name: count, dtype: int64


In [341]:
exact_drops = [
    # Administrative & Identifiers
    'Case #', 'Country', 'City', 'Site #', 'Date Created', 'General Comments',
    
    # Advanced Paramedic Interventions
    'Prehospital advanced airway', 'Prehospital drug administration', 
    'Epinephrine', 'Atropine', 'Amiodarone', 'Bicarbonate', 'Lidocaine', 'Dextrose',
    'Mechanical CPR device used by EMS/Private ambulance', "If 'Yes', please specify",
    "If 'Yes', please specify.1", "If 'Yes', please specify.2",
    
    # Granular/Redundant Timestamps
    'Time Ambulance left scene', 'Estimated time of arrest unknown',
    
    # ED & In-Hospital Interventions
    'Advanced airway used at ED', 'Drug administered at ED', 
    'Emergency PCI performed', 'Emergency CABG performed', 
    'Hypothermia therapy initiated', 'ECMO therapy initiated',
    'Epinephrine.1', 'Atropine.1', 'Amiodarone.1', 'Bicarbonate.1', 'Lidocaine.1', 'Dextrose.1'
]

# 2. Group the remaining categories using list comprehensions
# This dynamically finds any column in your dataframe that contains these phrases
medical_history_cols = [col for col in df.columns if 'Medical History' in col]
cpr_started_cols = [col for col in df.columns if 'Time CPR started' in col]
aed_applied_cols = [col for col in df.columns if 'Time AED applied' in col]
eq5d_cols = [col for col in df.columns if 'EQ-5D' in col]

# 3. Combine everything into one master "Burn List"
master_drop_list = exact_drops + medical_history_cols + cpr_started_cols + aed_applied_cols + eq5d_cols

# 4. Drop them from the dataframe
df_clean = df.drop(columns=master_drop_list, errors='ignore')

print(f"Original columns: {len(df.columns)}")
print(f"Cleaned columns: {len(df_clean.columns)}")
print(f"Total columns dropped: {len(df.columns) - len(df_clean.columns)}")

display(df_clean.head(20))
print(len(df_clean))

Original columns: 126
Cleaned columns: 71
Total columns dropped: 55


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Reason for discontinuing CPR at ED,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,EQ5D index,Date Last Saved,Year
0,EMS,4/1/2010,470146.0,NaN,Home Residence,HDB Level 7,60.0,Years,Male,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
1,EMS,4/1/2010,520926.0,NaN,Home Residence,HDB Level 2,60.0,Years,Female,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
2,EMS,4/1/2010,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,64.0,Years,Male,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,4/18/2012,2010.0
3,EMS,4/2/2010,680626.0,NaN,Home Residence,HDB Level 7,48.0,Years,Female,Chinese,...,ROSC,Admitted,Died in the hospital,4/5/2010,NaN,NaN,NaN,NaN,4/13/2012,2010.0
4,EMS,4/3/2010,468963.0,NaN,Place of Recreation,East Coast Park NSC Carpark,50.0,Years,Male,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
5,EMS,4/6/2010,460402.0,NaN,Home Residence,HDB Level 2,80.0,Years,Male,Malay,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
6,EMS,4/6/2010,310259.0,NaN,Home Residence,NaN,70.0,Years,Female,Indian,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,5/31/2013,2010.0
7,EMS,4/6/2010,161003.0,NaN,Home Residence,HDB Level 5,69.0,Years,Female,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
8,EMS,4/6/2010,681687.0,NaN,Home Residence,HDB Level 8,71.0,Years,Male,Chinese,...,Death,Died in ED,NaN,NaN,NaN,NaN,NaN,NaN,2/22/2011,2010.0
9,EMS,4/6/2010,120321.0,NaN,Home Residence,HDB Level 2,72.0,Years,Female,Indian,...,ROSC,Admitted,Died in the hospital,4/8/2010,NaN,NaN,NaN,NaN,2/22/2011,2010.0


28666


# Cleaning Columns and standardizing spelling

In [342]:
# List all columns that need cleaning
time_columns = [
    'Time call received at dispatch center', 'Time First Responder dispatched',
    'Time Ambulance dispatched', 'Time First Responder arrived at scene',
    'Time Ambulance arrived at scene', 'Time EMS arrived at patient side',
    'Time Ambulance arrived at ED', 'Estimated time of arrest',
    'Time CPR started by EMS/Private ambulance', 'Time AED applied by EMS/Private ambulance',
    'Time of first shock given', "If 'Yes', specify time", "If 'Yes', specify time.1",
    'Time of arrival at ED'
]

date_columns = ['Date of Incident', 'Date of arrival at ED', 'Date of Discharge or Death', 'Date Created']

yes_no_columns = [
    'Location Unknown', 'Medical History - No', 'Medical History - Unknown',
    'Medical History - Heart disease', 'Medical History - Diabetes', 'Medical History - Cancer',
    'Medical History - Hypertension', 'Medical History - Renal Disease', 'Medical History - Respiratory Disease',
    'Medical History - Hyperlipidemia', 'Medical History - Stroke', 'Medical History - HIV',
    'Medical History - Other', 'No First Responder dispatched', 'Estimated time of arrest unknown',
    'Bystander CPR', 'DA-CPR', 'Bystander AED applied', 'Resuscitation attempted by EMS/Private ambulance',
    'Time CPR started Unknown', 'Time AED applied Unknown', 'Prehospital Defibrillation',
    'Time of first shock Unknown', 'Defibrillation performed by - First Responder',
    'Defibrillation performed by - Ambulance Crew', 'Defibrillation performed by - Bystander - Healthcare provider',
    'Defibrillation performed by - Bystander - Lay Person', 'Defibrillation performed by - Bystander - Family',
    'Mechanical CPR device used by EMS/Private ambulance', 'Prehospital advanced airway',
    'Prehospital drug administration', 'Epinephrine', 'Atropine', 'Amiodarone', 'Lidocaine', 'Dextrose', 'Other',
    'Return of spontaneous circulation at scene/en-route', "Patient's status at ED arrival",
    'Patient status on arrival at ED - Pulse', 'Patient status on arrival at ED - Breathing',
    'ED Defibrillation performed', 'Mechanical CPR device used at ED', 'Advanced airway used at ED',
    'Drug administered at ED', 'Epinephrine.1', 'Atropine.1', 'Bicarbonate.1', 'Lidocaine.1', 'Dextrose.1', 'Other.1',
    'Return of spontaneous circulation at ED', 'ANY ROSC', 'Emergency PCI performed', 'Emergency CABG performed',
    'Hypothermia therapy initiated', 'ECMO therapy initiated', 'Patient neurological status - Unknown',
    'EQ-5D Unknown'
]

category_columns = [
    'Patient brought in by', 'Location Type', 'Location Type Other', 'Age Modifier', 'Gender', 'Race',
    'Arrest witnessed by', 'First CPR initiated by', 'First arrest rhythm', "If 'Yes', please specify",
    "If 'Yes', please specify.1", "If 'Yes', please specify.3", "If 'Yes', please specify.4",
    "If 'Non-Trauma', please specify", "If 'Non-Trauma', please specify.1", 'Level of destination hospital',
    'Destination Hospital', 'Cardiac rhythm on arrival at ED', 'Cause of arrest', 'Cause of arrest.1',
    'Reason for discontinuing CPR at ED', 'Outcome of patient', 'Patient status', 'Final status at scene'
]


In [343]:
# ============================================
# STANDARDIZED CLEANING FUNCTIONS
# ============================================

def clean_time_column(series):
    """Standardize any time column to HH:MM:SS format as time object"""
    def format_time(t):
        if pd.isna(t):
            return np.nan
        t_str = str(t).strip()
        parts = t_str.split(':')
        if len(parts) == 1:
            return f"{parts[0].zfill(2)}:00:00"
        elif len(parts) == 2:
            return f"{parts[0].zfill(2)}:{parts[1].zfill(2)}:00"
        elif len(parts) == 3:
            return f"{parts[0].zfill(2)}:{parts[1].zfill(2)}:{parts[2].zfill(2)}"
        return np.nan
    
    return pd.to_datetime(series.apply(format_time), format='%H:%M:%S', errors='coerce').dt.time

def clean_date_column(series):
    """Standardize date column to datetime (Singapore format: mixed)"""
    return pd.to_datetime(series, format='mixed', errors='coerce', dayfirst=True)

def clean_yes_no_column(series):
    """Standardize Yes/No columns to uppercase"""
    return series.str.strip().str.upper()

def clean_category_column(series):
    """Standardize categorical columns to uppercase"""
    return series.str.strip().str.upper()

In [344]:
print("Cleaning time columns...")
for col in time_columns:
    if col in df_clean.columns:
        df_clean[col] = clean_time_column(df_clean[col])

print("Cleaning date columns...")
for col in date_columns:
    if col in df_clean.columns:
        df_clean[col] = clean_date_column(df_clean[col])

print("Cleaning Yes/No columns...")
for col in yes_no_columns:
    if col in df_clean.columns:
        df_clean[col] = clean_yes_no_column(df_clean[col])

print("Cleaning category columns...")
for col in category_columns:
    if col in df_clean.columns:
        df_clean[col] = clean_category_column(df_clean[col])

# Special handling for columns with specific replacements
if 'Location Type' in df_clean.columns:
    df_clean['Location Type'] = df_clean['Location Type'].replace({'OTHERS': 'OTHER'})

if 'Race' in df_clean.columns:
    df_clean['Race'] = df_clean['Race'].replace({'OTHERS': 'OTHER', 'NAN': np.nan})

if 'Arrest witnessed by' in df_clean.columns:
    df_clean['Arrest witnessed by'] = df_clean['Arrest witnessed by'].replace({'AMBULANCE CREW': 'EMS/PRIVATE AMBULANCE'})

if 'ED Rosc Time Unknown' in df_clean.columns:
    df_clean['ED Rosc Time Unknown'] = clean_yes_no_column(df_clean['ED Rosc Time Unknown']).replace({'4:05': np.nan})

print("\n✅ All columns cleaned successfully!")
display(df_clean.head(3))
display(len(df_clean))

Cleaning time columns...
Cleaning date columns...
Cleaning Yes/No columns...
Cleaning category columns...

✅ All columns cleaned successfully!


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Reason for discontinuing CPR at ED,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,EQ5D index,Date Last Saved,Year
0,EMS,2010-01-04,470146.0,NaN,HOME RESIDENCE,HDB LEVEL 7,60.0,YEARS,MALE,CHINESE,...,DEATH,DIED IN ED,NaN,NaT,NaN,NaN,NaN,NaN,2/22/2011,2010.0
1,EMS,2010-01-04,520926.0,NaN,HOME RESIDENCE,HDB LEVEL 2,60.0,YEARS,FEMALE,CHINESE,...,DEATH,DIED IN ED,NaN,NaT,NaN,NaN,NaN,NaN,2/22/2011,2010.0
2,EMS,2010-01-04,560565.0,NaN,HEALTHCARE FACILITY,NKF DIALYSIS CENTRE,64.0,YEARS,MALE,CHINESE,...,DEATH,DIED IN ED,NaN,NaT,NaN,NaN,NaN,NaN,4/18/2012,2010.0


28666

In [345]:
print("--- Temporal Filter")
print(df_clean['Date of Incident'].dt.year.value_counts(dropna=False))

# print("--- Age Filter")
# print(df_clean['Age'].value_counts(dropna=False))

print("--- Etiology Filter")
etiology_filter = df_clean[
    df_clean["If 'Non-Trauma', please specify"].isin(['PRESUMED CARDIAC ETIOLOGY']) | 
    df_clean["If 'Non-Trauma', please specify"].isna()
]
print(etiology_filter["If 'Non-Trauma', please specify"].value_counts(dropna=False))

print("--- Witness Filter ---")
print(df_clean['Arrest witnessed by'].value_counts(dropna=False))

print("\n--- Rhythm Filter ---")
print(df_clean['First arrest rhythm'].value_counts(dropna=False))

print("\n--- Resuscitation Attempted Filter ---")
print(df_clean['Resuscitation attempted by EMS/Private ambulance'].value_counts(dropna=False))

--- Temporal Filter
Date of Incident
2021.0    3636
2020.0    3431
2019.0    3233
2018.0    2975
2017.0    2841
2016.0    2505
2015.0    2372
2014.0    2038
2013.0    1736
2012.0    1440
2011.0    1377
2010.0    1081
NaN          1
Name: count, dtype: int64
--- Etiology Filter
If 'Non-Trauma', please specify
NaN                          27916
PRESUMED CARDIAC ETIOLOGY      561
Name: count, dtype: int64
--- Witness Filter ---
Arrest witnessed by
NOT WITNESSED                      12804
BYSTANDER - FAMILY                  8191
BYSTANDER - LAY PERSON              3891
EMS/PRIVATE AMBULANCE               2538
BYSTANDER - HEALTHCARE PROVIDER     1241
NaN                                    1
Name: count, dtype: int64

--- Rhythm Filter ---
First arrest rhythm
ASYSTOLE                      14842
PEA                            7822
VF                             3698
UNKNOWN UNSHOCKABLE RHYTHM      985
UNKNOWN SHOCKABLE RHYTHM        711
UNKNOWN                         264
NaN                 

In [346]:
print("\n--- COMBINED DAG FLOWCHART & EXCLUSION CRITERIA ---")

# Ensure 'Date of Incident' is daytime so we can extract year safely
df_clean['Year'] = df_clean['Date of Incident'].dt.year

# 1. Base Exclusions (Temporal 2010-2017 & Etiology) before we start the reporting loop
print("Applying Base Filters: Restricting to 2010-2017 & Presumed Cardiac Etiology...")
base_filtered = df_clean[df_clean['Year'].isin([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017])].copy()

base_filtered = base_filtered[
    base_filtered["If 'Non-Trauma', please specify"].isin(['PRESUMED CARDIAC ETIOLOGY']) | 
    base_filtered["If 'Non-Trauma', please specify"].isna()
]

print(f"Base Dataset Size After Initial Filters: N = {len(base_filtered)}")

# We will collect the final filtered dataframe for each year here to assemble the complete Utstein cohort at the end
final_cohort_dfs = []

for year in sorted(base_filtered['Year'].dropna().unique()):
    year_df = base_filtered[base_filtered['Year'] == year]
    
    print(f"\n=== YEAR {int(year)} (Initial N={len(year_df)}) ===")
    
    # ---------------------------------------------------------
    # STEP 1: Witnessed Status (of ALL cases that year)
    # ---------------------------------------------------------
    bystander_mask = year_df['Arrest witnessed by'].astype(str).str.contains('BYSTANDER', case=False, na=False)
    bystander_witnesses = bystander_mask.sum()
    not_witnessed = (year_df['Arrest witnessed by'] == 'NOT WITNESSED').sum()
    ems_witnessed = (year_df['Arrest witnessed by'] == 'EMS/PRIVATE AMBULANCE').sum()
    unknown_witnessed = len(year_df) - (bystander_witnesses + not_witnessed + ems_witnessed)

    print("1. Witnessed Status:")
    print(f"  Not Witnessed: {not_witnessed}")
    print(f"  Bystander Witnesses: {bystander_witnesses}")
    print(f"  EMS Witnessed: {ems_witnessed}")
    print(f"  Unknown/Other: {unknown_witnessed}")

    # FILTER: Move forward only with Bystander Witnessed
    year_df = year_df[bystander_mask].copy()
    
    # ---------------------------------------------------------
    # STEP 2: First Rhythm (of ONLY Bystander Witnessed)
    # OPTIONAL HYBRID LOGIC: If 'UNKNOWN' is manually backfilled by 'Cardiac rhythm on arrival at ED'
    # ---------------------------------------------------------
    
    # Create a staging column for calculation that backfills 'UNKNOWN' with ED rhythm 
    # (This matches the exact missing cases from 2011 without affecting true NaNs)
    combined_rhythm = np.where(
        year_df['First arrest rhythm'] == 'UNKNOWN', 
        year_df['Cardiac rhythm on arrival at ED'], 
        year_df['First arrest rhythm']
    )
    
    shockable_mask = pd.Series(combined_rhythm).astype(str).str.contains('VF|VT|UNKNOWN SHOCKABLE', case=False, na=False).values
    shockable = shockable_mask.sum()
    
    unshockable = pd.Series(combined_rhythm).astype(str).str.contains('ASYSTOLE|PEA|UNKNOWN UNSHOCKABLE', case=False, na=False).sum()
    unknown_rhythm = len(year_df) - (shockable + unshockable)

    print(f"\n2. First Rhythm (Out of {len(year_df)} Bystander Witnessed):")
    print(f"  Shockable: {shockable}")
    print(f"  Unshockable: {unshockable}")
    print(f"  Unknown: {unknown_rhythm}")

    # FILTER: Move forward only with Shockable
    year_df = year_df[shockable_mask].copy()

    # ---------------------------------------------------------
    # STEP 3: Outcomes (of ONLY Shockable & Bystander Witnessed)
    # ---------------------------------------------------------
    died_scene = (year_df['Final status at scene'] == 'PRONOUNCED DEAD AT SCENE')
    died_ed = (year_df['Outcome of patient'] == 'DIED IN ED')
    died_hosp = year_df['Patient status'].astype(str).str.contains('DIED|DEATH', case=False, na=False)
    
    died = (died_scene | died_ed | died_hosp).sum()
    discharged_alive = (year_df['Patient status'] == 'DISCHARGED ALIVE').sum()
    in_hospital_30 = (year_df['Patient status'] == 'REMAINS IN HOSPITAL AT 30TH DAY POST ARREST').sum()
    unknown_outcome = len(year_df) - (died + discharged_alive + in_hospital_30)

    print(f"\n3. Outcome Counts (Out of {len(year_df)} Shockable Bystander cases):")
    print(f"  Died: {died}")
    print(f"  Discharged alive: {discharged_alive}")
    print(f"  In hospital after 30 days: {in_hospital_30}")
    print(f"  Unknown/Other: {unknown_outcome}")
    print("-" * 50)
    
    # Save this year's fully filtered cohort
    final_cohort_dfs.append(year_df)

# Combine all the year-by-year dataframes back into a single main Utstein cohort
utstein_filtered = pd.concat(final_cohort_dfs, ignore_index=True)
print(f"\nFINAL STRICT UTSTEIN COHORT CREATED: N = {len(utstein_filtered)} CASES")


--- COMBINED DAG FLOWCHART & EXCLUSION CRITERIA ---
Applying Base Filters: Restricting to 2010-2017 & Presumed Cardiac Etiology...
Base Dataset Size After Initial Filters: N = 15386

=== YEAR 2010 (Initial N=1081) ===
1. Witnessed Status:
  Not Witnessed: 564
  Bystander Witnesses: 439
  EMS Witnessed: 78
  Unknown/Other: 0

2. First Rhythm (Out of 439 Bystander Witnessed):
  Shockable: 100
  Unshockable: 321
  Unknown: 18

3. Outcome Counts (Out of 100 Shockable Bystander cases):
  Died: 87
  Discharged alive: 12
  In hospital after 30 days: 1
  Unknown/Other: 0
--------------------------------------------------

=== YEAR 2011 (Initial N=1377) ===
1. Witnessed Status:
  Not Witnessed: 490
  Bystander Witnesses: 775
  EMS Witnessed: 112
  Unknown/Other: 0

2. First Rhythm (Out of 775 Bystander Witnessed):
  Shockable: 183
  Unshockable: 586
  Unknown: 6

3. Outcome Counts (Out of 183 Shockable Bystander cases):
  Died: 161
  Discharged alive: 19
  In hospital after 30 days: 3
  Unknow

# Save newly cleaned PAROS dataset

In [347]:
utstein_filtered.to_csv(CLEANED_DATASET_PATH, index=False)
df = pd.read_csv(CLEANED_DATASET_PATH)

display(df.head(3))
display(len(df))

,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Reason for discontinuing CPR at ED,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,EQ5D index,Date Last Saved,Year
0,EMS,2010-03-05,689101.0,NaN,HOME RESIDENCE,36 CHOA CHU KANG ST 64 #15-02 THE QUINTET,68.0,YEARS,MALE,CHINESE,...,ROSC,ADMITTED,DISCHARGED ALIVE,2010-05-28,1.0,1.0,NaN,NaN,8/31/2011,2010.0
1,EMS,2010-06-14,400002.0,NaN,HOME RESIDENCE,NaN,87.0,YEARS,FEMALE,CHINESE,...,DEATH,DIED IN ED,NaN,NaN,NaN,NaN,NaN,NaN,5/23/2013,2010.0
2,EMS,2010-06-16,650210.0,NaN,PUBLIC/COMMERCIAL BUILDING,NaN,75.0,YEARS,MALE,CHINESE,...,DEATH,DIED IN ED,NaN,NaN,NaN,NaN,NaN,NaN,11/9/2012,2010.0


1808